[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fisica_No_Lineal_y_Sistemas_Complejos_%28NLD%29/Fisica_Biologica_%28q-bio%29/NLD-08_Simulacion_Neurona_Hodgkin_Huxley.ipynb)

# [NLD-08] Neurofísica: Potenciales de Acción Neuronal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Modelo de Hodgkin-Huxley para el Potencial de Acción Neuronal

# Parámetros de la membrana (Calamar Gigante)
C_m = 1.0     # Capacitancia de membrana (uF/cm^2)
g_Na = 120.0  # Conductancia máxima Na+ (mS/cm^2)
g_K = 36.0    # Conductancia máxima K+ (mS/cm^2)
g_L = 0.3     # Conductancia Leak (mS/cm^2)
E_Na = 50.0   # Potencial Nernst Na+ (mV)
E_K = -77.0   # Potencial Nernst K+ (mV)
E_L = -54.387 # Potencial Leak (mV)

# Funciones de compuerta (Gate dynamics) dependientes del voltaje
def alpha_m(V): return 0.1*(V+40.0)/(1.0 - np.exp(-(V+40.0)/10.0))
def beta_m(V):  return 4.0*np.exp(-(V+65.0)/18.0)
def alpha_h(V): return 0.07*np.exp(-(V+65.0)/20.0)
def beta_h(V):  return 1.0/(1.0 + np.exp(-(V+35.0)/10.0))
def alpha_n(V): return 0.01*(V+55.0)/(1.0 - np.exp(-(V+55.0)/10.0))
def beta_n(V):  return 0.125*np.exp(-(V+65)/80.0)

# Sistema Diferencial HH
def hodgkin_huxley(Y, t, I_inj):
    V, m, h, n = Y
    
    # Corrientes iónicas
    I_Na = g_Na * (m**3) * h * (V - E_Na)
    I_K  = g_K  * (n**4) * (V - E_K)
    I_L  = g_L  * (V - E_L)
    
    dVdt = (I_inj(t) - I_Na - I_K - I_L) / C_m
    dmdt = alpha_m(V)*(1.0 - m) - beta_m(V)*m
    dhdt = alpha_h(V)*(1.0 - h) - beta_h(V)*h
    dndt = alpha_n(V)*(1.0 - n) - beta_n(V)*n
    return [dVdt, dmdt, dhdt, dndt]

# Corriente inyectada externa (Pulsos)
def I_ext(t):
    if 10 <= t <= 12 or 25 <= t <= 35:
        return 15.0 # uA/cm^2 (Suficiente para disparar el potencial)
    return 0.0

time = np.linspace(0, 50, 5000)
Y0 = [-65.0, 0.05, 0.6, 0.32] # Estado de reposo [V, m, h, n]

sol = odeint(hodgkin_huxley, Y0, time, args=(I_ext,))
V_membrana = sol[:, 0]

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(time, V_membrana, 'k-', lw=2)
plt.ylabel("Potencial V_m (mV)")
plt.title("Potenciales de Acción (Spikes) en el Modelo de Hodgkin-Huxley")
plt.grid()

plt.subplot(2, 1, 2)
I_plot = [I_ext(t) for t in time]
plt.plot(time, I_plot, 'orange', lw=2)
plt.xlabel("Tiempo (ms)")
plt.ylabel("Corriente Inyectada $I_{ext}$")
plt.grid()

plt.tight_layout()
plt.show()

